In [1]:
import os, platform, shutil
print("Python:", platform.python_version())
print("System:", platform.platform())
total, used, free = shutil.disk_usage("/content")
print("Runtime disk total (GB):", round(total / 1024**3, 2))
print("Runtime disk free (GB):", round(free / 1024**3, 2))


Python: 3.12.13
System: Linux-6.6.122+-x86_64-with-glibc2.35
Runtime disk total (GB): 107.72
Runtime disk free (GB): 87.73


In [2]:
import os, subprocess, time
IDF_VERSION = "v5.5.4"
IDF_PATH = "/content/esp-idf"

if os.path.exists(IDF_PATH):
    subprocess.run(["rm", "-rf", IDF_PATH], check=True)

start = time.time()
result = subprocess.run(
    ["git", "clone", "--branch", IDF_VERSION, "--depth", "1",
     "--recursive", "--shallow-submodules",
     "https://github.com/espressif/esp-idf.git", IDF_PATH],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout[-4000:])
if result.returncode != 0:
    raise RuntimeError("ESP-IDF download failed")
print("Elapsed minutes:", round((time.time() - start) / 60, 2))

97bc7a7 -> FETCH_HEAD
Submodule path 'components/cmock/CMock/vendor/unity': checked out 'cf949f45ca6d172a177b00da21310607b97bc7a7'
From https://github.com/espressif/esp-coex-lib
 * branch            ee5cd79583c02f23e43e62931ffb55f5a4992d0f -> FETCH_HEAD
Submodule path 'components/esp_coex/lib': checked out 'ee5cd79583c02f23e43e62931ffb55f5a4992d0f'
From https://github.com/espressif/esp-phy-lib
 * branch            3d57415af6e4c92eff2c4c3463e20a51d7340aba -> FETCH_HEAD
Submodule path 'components/esp_phy/lib': checked out '3d57415af6e4c92eff2c4c3463e20a51d7340aba'
From https://github.com/espressif/esp32-wifi-lib
 * branch            b87cce81803b85c3639fb14bac6c24cff6d0fbad -> FETCH_HEAD
Submodule path 'components/esp_wifi/lib': checked out 'b87cce81803b85c3639fb14bac6c24cff6d0fbad'
From https://github.com/espressif/tlsf
 * branch            2867f6883a12920b1969ff9624c0ab0e4185c2ce -> FETCH_HEAD
Submodule path 'components/heap/tlsf': checked out '2867f6883a12920b1969ff9624c0ab0e4185c2ce'


In [3]:
import subprocess, os, shutil, time

subprocess.run(
    ["bash", "-lc",
     "apt-get update -qq && apt-get install -y python3.12-venv python3-venv python3-pip ninja-build"],
    check=True
)

py_env = "/root/.espressif/python_env/idf5.5_py3.12_env"
if os.path.exists(py_env):
    shutil.rmtree(py_env)

result = subprocess.run(
    ["bash", "-lc", f"cd {IDF_PATH} && ./install.sh esp32,esp32s3"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout[-8000:])
if result.returncode != 0:
    raise RuntimeError("ESP-IDF tool installation failed")


irements/requirements.core.txt (line 18))
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 29.9 MB/s  0:00:00
  Created wheel for esptool: filename=esptool-4.12.0-py3-none-any.whl size=616537 sha256=9fafd3b419db100044637be9f795c9aa2126e62bb5c7ccc80165fb51d99eb692
  Stored in directory: /root/.cache/pip/wheels/8e/95/c0/03b414199505d3d22b6a9843a9e3cb2db9d72673433364f6f4
Successfully built esptool

Upgrading pip...
Upgrading setuptools...
Installing Python packages
 Constraint file: /root/.espressif/espidf.constraints.v5.5.txt
 Requirement files:
  - /content/esp-idf/tools/requirements/requirements.core.txt
All done! You can now run:

  . ./export.sh




In [4]:
import os, shutil, subprocess, time, json

SOURCE_PROJECT = f"{IDF_PATH}/examples/get-started/hello_world"
WORK_ROOT = "/content/tactic_pilot"
RESULTS_DIR = f"{WORK_ROOT}/results"

shutil.rmtree(WORK_ROOT, ignore_errors=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

projects = {
    "esp32": f"{WORK_ROOT}/hello_esp32",
    "esp32s3": f"{WORK_ROOT}/hello_esp32s3"
}
for p in projects.values():
    shutil.copytree(SOURCE_PROJECT, p)

def build_target(target, project_path, prefix):
    log_path = f"{RESULTS_DIR}/{prefix}_{target}.log"
    cmd = f'''
    set -o pipefail
    source {IDF_PATH}/export.sh >/dev/null 2>&1
    cd {project_path}
    idf.py set-target {target}
    idf.py build
    '''
    start = time.time()
    result = subprocess.run(["bash", "-lc", cmd], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = round(time.time() - start, 2)
    open(log_path, "w", encoding="utf-8").write(result.stdout)
    return {
        "target": target,
        "status": "PASS" if result.returncode == 0 else "FAIL",
        "exit_code": result.returncode,
        "elapsed_seconds": elapsed,
        "log_file": log_path
    }

baseline_results = [build_target(t, p, "baseline") for t, p in projects.items()]
baseline = {
    "esp_idf_version": "v5.5.4",
    "sample_project": "examples/get-started/hello_world",
    "evaluation_stage": "clean baseline",
    "results": baseline_results
}
open(f"{RESULTS_DIR}/baseline_summary.json", "w").write(json.dumps(baseline, indent=2))
print(json.dumps(baseline, indent=2))


{
  "esp_idf_version": "v5.5.4",
  "sample_project": "examples/get-started/hello_world",
  "evaluation_stage": "clean baseline",
  "results": [
    {
      "target": "esp32",
      "status": "PASS",
      "exit_code": 0,
      "elapsed_seconds": 69.7,
      "log_file": "/content/tactic_pilot/results/baseline_esp32.log"
    },
    {
      "target": "esp32s3",
      "status": "PASS",
      "exit_code": 0,
      "elapsed_seconds": 73.62,
      "log_file": "/content/tactic_pilot/results/baseline_esp32s3.log"
    }
  ]
}


In [5]:
from pathlib import Path
import os, shutil, subprocess, time, json

TASK_ROOT = f"{WORK_ROOT}/task_01_corrected"
shutil.rmtree(TASK_ROOT, ignore_errors=True)

task_projects = {
    "esp32": f"{TASK_ROOT}/changed_esp32",
    "esp32s3": f"{TASK_ROOT}/changed_esp32s3"
}
for p in task_projects.values():
    shutil.copytree(SOURCE_PROJECT, p)

CONTROLLED_DEFECT = r'''
/*
 * TACTIC Task 01: controlled target-specific build defect.
 * Expected:
 *   ESP32    -> FAIL
 *   ESP32-S3 -> PASS
 */
#include "sdkconfig.h"

#if defined(CONFIG_IDF_TARGET_ESP32) && CONFIG_IDF_TARGET_ESP32
#error "TACTIC_TASK_01: controlled ESP32-only build failure"
#endif
'''

def find_app_main(project_path):
    for candidate in Path(project_path).rglob("*.c"):
        if "app_main" in candidate.read_text(encoding="utf-8", errors="ignore"):
            return candidate
    raise FileNotFoundError("app_main source not found")

for target, project_path in task_projects.items():
    source_file = find_app_main(project_path)
    original = source_file.read_text(encoding="utf-8")
    source_file.write_text(CONTROLLED_DEFECT + "\n" + original, encoding="utf-8")

def configure(target, project_path):
    result = subprocess.run(
        ["bash", "-lc", f'''
        set -o pipefail
        source {IDF_PATH}/export.sh >/dev/null 2>&1
        cd {project_path}
        idf.py set-target {target}
        grep '^CONFIG_IDF_TARGET=' sdkconfig || true
        grep '^CONFIG_IDF_TARGET_ESP32=' sdkconfig || true
        grep '^CONFIG_IDF_TARGET_ESP32S3=' sdkconfig || true
        '''],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Configuration failed for {target}")

for target, project_path in task_projects.items():
    configure(target, project_path)

def build_changed(target, project_path):
    log_path = f"{RESULTS_DIR}/task_01_corrected_changed_{target}.log"
    start = time.time()
    result = subprocess.run(
        ["bash", "-lc", f'''
        set -o pipefail
        source {IDF_PATH}/export.sh >/dev/null 2>&1
        cd {project_path}
        idf.py build
        '''],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    elapsed = round(time.time() - start, 2)
    open(log_path, "w", encoding="utf-8").write(result.stdout)
    marker = "TACTIC_TASK_01: controlled ESP32-only build failure"
    return {
        "target": target,
        "status": "PASS" if result.returncode == 0 else "FAIL",
        "exit_code": result.returncode,
        "elapsed_seconds": elapsed,
        "expected_status": "FAIL" if target == "esp32" else "PASS",
        "failure_marker_found": marker in result.stdout,
        "log_file": log_path
    }

esp32 = build_changed("esp32", task_projects["esp32"])
esp32s3 = build_changed("esp32s3", task_projects["esp32s3"])

supported = (
    esp32["status"] == "FAIL"
    and esp32["failure_marker_found"]
    and esp32s3["status"] == "PASS"
)

evidence = {
    "task_id": "TACTIC_TASK_01_CORRECTED",
    "task_type": "controlled target-specific build defect",
    "claim": "The injected change breaks the ESP32 build while preserving the ESP32-S3 build.",
    "environment": {
        "esp_idf_version": "v5.5.4",
        "sample_project": "examples/get-started/hello_world",
        "targets": ["esp32", "esp32s3"]
    },
    "baseline": {"esp32": "PASS", "esp32s3": "PASS"},
    "changed_results": {"esp32": esp32, "esp32s3": esp32s3},
    "admission_decision": "ADMIT" if supported else "ABSTAIN",
    "claim_supported": supported
}

open(f"{RESULTS_DIR}/task_01_corrected_evidence.json", "w").write(
    json.dumps(evidence, indent=2)
)
print(json.dumps(evidence, indent=2))

Adding "set-target"'s dependency "fullclean" to list of commands with default set of options.
Executing action: fullclean
Build directory '/content/tactic_pilot/task_01_corrected/changed_esp32/build' not found. Nothing to clean.
Executing action: set-target
Set Target to: esp32, new sdkconfig will be created.
Running cmake in directory /content/tactic_pilot/task_01_corrected/changed_esp32/build
Executing "cmake -G Ninja -DPYTHON_DEPS_CHECKED=1 -DPYTHON=/root/.espressif/python_env/idf5.5_py3.12_env/bin/python -DESP_PLATFORM=1 -DIDF_TARGET=esp32 -DCCACHE_ENABLE=0 /content/tactic_pilot/task_01_corrected/changed_esp32"...
-- Found Git: /usr/bin/git (found version "2.34.1")
-- Minimal build - ON
-- The C compiler identification is GNU 14.2.0
-- The CXX compiler identification is GNU 14.2.0
-- The ASM compiler identification is GNU
-- Found assembler: /root/.espressif/tools/xtensa-esp-elf/esp-14.2.0_20260121/xtensa-esp-elf/bin/xtensa-esp32-elf-gcc
-- Detecting C compiler ABI info
-- Detectin

In [6]:
# Optional: package the generated logs and JSON files for download.
import shutil
archive = shutil.make_archive("/content/TACTIC_Pilot_Runtime_Results", "zip",
                              "/content/tactic_pilot/results")
print(archive)


/content/TACTIC_Pilot_Runtime_Results.zip
